# 05 · What kind of practice lines up with midterm scores?

All comparisons are made **within cohort** (semester × midterm), because exam difficulty varies a lot between terms.
1. Screen every numeric feature (116) with within-cohort rank correlations, controlling the false discovery rate.
2. Quantify the headline (Parsons first-try success) with a quartile gap, a student-clustered bootstrap, and fixed-effects regressions.
3. Check it doesn't hinge on the inferred answer keys.

These are **associations**: students who solve Parsons problems on the first try may simply be stronger students.
The midterm-2-controlling-for-midterm-1 model is the closest this data gets to separating the two.

In [1]:
import pandas as pd
from so_ell import config
from so_ell.notebook import show, show_sql, describe
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
config.ensure_dirs()
T = config.TABLES_DIR

In [2]:
from so_ell import stats, pipeline
from so_ell.features import mid1_to_mid2_changes
features = stats.add_cohort(pd.read_parquet(config.PROCESSED_DIR / "student_exam_features.parquet"))
print(f"{len(features):,} student-exams in {features['cohort'].nunique()} cohorts")

2,745 student-exams in 16 cohorts


## 1. Screening every feature

In [3]:
show(stats.within_group_rank_corr, stats.within_group_quartile_gap, stats.benjamini_hochberg, stats.compute_corr_table)
corr = stats.compute_corr_table(features, "cohort", "score_pct", pipeline.numeric_feature_cols(features))
corr.to_csv(T / "feature_signals.csv", index=False)
print(f"{(corr['q_value_bh'] < 0.05).sum()} of {len(corr)} features significant at q < 0.05")
corr.head(15)[["feature", "n", "within_group_rank_corr", "top_quartile_minus_bottom_quartile_pct_points", "q_value_bh"]].round(4)

`so_ell.stats.within_group_rank_corr`
```python
def within_group_rank_corr(df: pd.DataFrame, group_col: str, feature: str, target: str) -> float:
    """Pearson correlation of within-cohort percentile ranks (a pooled Spearman)."""
    use = df[[group_col, feature, target]].dropna().copy()
    if len(use) < 10 or use[feature].nunique() < 3 or use[target].nunique() < 3:
        return np.nan
    use["x_rank"] = use.groupby(group_col)[feature].rank(pct=True)
    use["y_rank"] = use.groupby(group_col)[target].rank(pct=True)
    return float(use[["x_rank", "y_rank"]].corr().iloc[0, 1])
```

`so_ell.stats.within_group_quartile_gap`
```python
def within_group_quartile_gap(df: pd.DataFrame, group_col: str, feature: str, target: str) -> float:
    """Mean target in the top feature quartile minus the bottom one, quartiles cut per cohort."""
    use = df[[group_col, feature, target]].dropna().copy()
    if len(use) < 20 or use[feature].nunique() < 4:
        return np.nan
    use["feature_rank"] = use.groupby(group_col)[feature].rank(pct=True, method="average")
    bottom = use.loc[use["feature_rank"] <= 0.25, target]
    top = use.loc[use["feature_rank"] > 0.75, target]
    if bottom.empty or top.empty:
        return np.nan
    return float(top.mean() - bottom.mean())
```

`so_ell.stats.benjamini_hochberg`
```python
def benjamini_hochberg(p_values: pd.Series) -> pd.Series:
    p = p_values.to_numpy(dtype=float)
    order = np.argsort(p)
    ranked = p[order] * len(p) / np.arange(1, len(p) + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = np.clip(q, 0, 1)
    return pd.Series(out, index=p_values.index)
```

`so_ell.stats.compute_corr_table`
```python
def compute_corr_table(df: pd.DataFrame, group_col: str, target: str, features: list[str]) -> pd.DataFrame:
    rows = []
    for feature in features:
        use = df[[group_col, feature, target]].dropna()
        if use.empty:
            continue
        r = within_group_rank_corr(df, group_col, feature, target)
        n = len(use)
        # t-approximation for a rank correlation; good enough for screening.
        t = r * np.sqrt((n - 2) / max(1e-12, 1 - r**2)) if pd.notna(r) else np.nan
        rows.append(
            {
                "feature": feature,
                "n": int(n),
                "within_group_rank_corr": r,
                "p_value": float(2 * sps.t.sf(abs(t), n - 2)) if pd.notna(t) else np.nan,
                "top_quartile_minus_bottom_quartile_pct_points": within_group_quartile_gap(df, group_col, feature, target),
            }
        )
    out = pd.DataFrame(rows).dropna(subset=["within_group_rank_corr"])
    if not out.empty:
        out["q_value_bh"] = benjamini_hochberg(out["p_value"])
    return out.sort_values("within_group_rank_corr", ascending=False).reset_index(drop=True)
```

94 of 116 features significant at q < 0.05


,feature,n,within_group_rank_corr,top_quartile_minus_bottom_quartile_pct_points,q_value_bh
0,parsons__mean_first_score,2739,0.5703,20.3901,0.0
1,parsons__avg_event_score,2739,0.5648,20.8181,0.0
2,learning__mean_first_score,2741,0.5300,19.1860,0.0
3,all_activity__avg_event_score,2741,0.5183,19.4736,0.0
4,mchoice__mean_first_score,2741,0.4706,17.6289,0.0
5,mchoice__avg_event_score,2741,0.4294,16.2664,0.0
6,fillb__mean_first_score,2739,0.3556,12.4836,0.0
7,mix__activecode_share,2741,0.3339,13.3455,0.0
8,dragndrop__mean_first_score,2728,0.3296,12.1453,0.0
9,parsons__mean_last_score,2739,0.3015,11.4375,0.0


Volume of activity is a weak signal, and raw event counts are even negative (more repeated wrong checks), compared with the *quality* of first attempts:

In [4]:
corr[corr.feature.str.contains("total_events|active_days")][["feature", "within_group_rank_corr", "q_value_bh"]].round(4)

,feature,within_group_rank_corr,q_value_bh
32,shortanswer__active_days,0.1014,0.0000
36,mchoice__active_days,0.0784,0.0001
38,activecode__total_events,0.0673,0.0007
46,dragndrop__active_days,0.0616,0.0019
47,shortanswer__total_events,0.0596,0.0027
63,fillb__active_days,0.0330,0.0980
67,all_activity__active_days,0.0237,0.2401
68,parsons__active_days,0.0229,0.2534
69,consistency__active_days,0.0215,0.2844
70,activecode__active_days,0.0199,0.3232


## 2. How big is the Parsons gap, and how sure are we?
Students (not rows) are resampled, because one student contributes two midterms.

In [5]:
show(stats.bootstrap_quartile_gap)
boot = stats.bootstrap_quartile_gap(features, n_boot=1000)
pd.DataFrame([boot]).to_csv(T / "parsons_gap_bootstrap.csv", index=False)
pd.DataFrame([boot]).round(2)

`so_ell.stats.bootstrap_quartile_gap`
```python
def bootstrap_quartile_gap(
    df: pd.DataFrame,
    feature: str = PARSONS,
    target: str = "score_pct",
    n_boot: int = 1000,
    seed: int = RANDOM_SEED,
) -> dict[str, float]:
    """95% CI for the quartile gap, resampling students (clusters) with replacement."""
    use = add_cohort(df)[["cohort", "semester_raw", "anon_student_id", feature, target]].dropna()
    use["cluster"] = use["semester_raw"].astype(str) + "|" + use["anon_student_id"].astype(str)
    use = use.reset_index(drop=True)
    rows_by_cluster = list(use.groupby("cluster").indices.values())
    rng = np.random.default_rng(seed)
    draws = []
    for _ in range(n_boot):
        picks = rng.integers(0, len(rows_by_cluster), size=len(rows_by_cluster))
        boot = use.iloc[np.concatenate([rows_by_cluster[i] for i in picks])]
        draws.append(within_group_quartile_gap(boot, "cohort", feature, target))
    draws = np.array([d for d in draws if pd.notna(d)])
    return {
        "estimate": within_group_quartile_gap(use, "cohort", feature, target),
        "ci_low": float(np.percentile(draws, 2.5)),
        "ci_high": float(np.percentile(draws, 97.5)),
        "n_boot": len(draws),
    }
```

,estimate,ci_low,ci_high,n_boot
0,20.39,18.78,22.13,1000


## 3. Fixed-effects regressions
Score on Parsons first-try %, with cohort fixed effects and student-clustered standard errors. Spec 3 asks: among students with the *same* midterm 1 score, does Parsons practice before midterm 2 still line up with midterm 2?

In [6]:
show(stats._prep_model_frame, stats._fit_clustered, stats.fit_parsons_models)
fe = stats.fit_parsons_models(features)
fe.to_csv(T / "parsons_models.csv", index=False)
fe[["spec", "n", "coef", "std_err", "ci_low", "ci_high", "p_value", "r_squared"]].round(4)

`so_ell.stats._prep_model_frame`
```python
def _prep_model_frame(df: pd.DataFrame) -> pd.DataFrame:
    use = add_cohort(df).rename(
        columns={
            PARSONS: "parsons_first",
            "all_activity__total_events": "total_events",
            "all_activity__active_days": "active_days",
            "coverage__unique_chapters": "unique_chapters",
            "timing__share_last_7d": "share_last_7d",
        }
    )
    use = use.dropna(subset=["score_pct", "parsons_first"]).copy()
    # 0-1 -> percentage points, so the coefficient reads "exam points per Parsons point".
    use["parsons_first"] = use["parsons_first"] * 100
    use["student"] = use["semester_raw"].astype(str) + "|" + use["anon_student_id"].astype(str)
    for col in ["total_events", "active_days", "unique_chapters", "share_last_7d"]:
        if col in use:
            use[col] = pd.to_numeric(use[col], errors="coerce")
            use[col] = use[col].fillna(use[col].median())
    return use
```

`so_ell.stats._fit_clustered`
```python
def _fit_clustered(formula: str, data: pd.DataFrame):
    groups = pd.factorize(data["student"])[0]
    return smf.ols(formula, data=data).fit(cov_type="cluster", cov_kwds={"groups": groups})
```

`so_ell.stats.fit_parsons_models`
```python
def fit_parsons_models(df: pd.DataFrame) -> pd.DataFrame:
    """Parsons first-try quality vs exam %, three nested specifications."""
    use = _prep_model_frame(df)
    rows = []

    m1 = _fit_clustered("score_pct ~ parsons_first + C(cohort)", use)
    rows.append(
        _ols_row(m1, "cohort_fe", "parsons_first", len(use), use["student"].nunique(),
                 "Exam points per 1-point higher Parsons first-try %; semester x exam fixed effects; SEs clustered by student.")
    )

    controls = [c for c in ["active_days", "unique_chapters", "share_last_7d"] if c in use]
    rhs = ["parsons_first", "np.log1p(total_events)"] + controls
    m2 = _fit_clustered("score_pct ~ " + " + ".join(rhs) + " + C(cohort)", use)
    rows.append(
        _ols_row(m2, "cohort_fe_plus_effort", "parsons_first", len(use), use["student"].nunique(),
                 "Adds log events, active days, chapter coverage and last-week share as effort controls.")
    )

    # Prior ability: for midterm 2, control for the same student's midterm 1 score.
    mid1 = use.loc[use["midterm"].eq("mid1"), ["student", "score_pct"]].rename(columns={"score_pct": "mid1_score"})
    mid2 = use.loc[use["midterm"].eq("mid2")].merge(mid1, on="student", how="inner")
    if len(mid2) > 50:
        m3 = _fit_clustered("score_pct ~ parsons_first + mid1_score + np.log1p(total_events) + C(cohort)", mid2)
        rows.append(
            _ols_row(m3, "mid2_controls_for_mid1", "parsons_first", len(mid2), mid2["student"].nunique(),
                     "Midterm 2 only, controlling for the student's own midterm 1 score (prior ability) and volume.")
        )
    return pd.DataFrame(rows)
```

,spec,n,coef,std_err,ci_low,ci_high,p_value,r_squared
0,cohort_fe,2739,0.4868,0.0192,0.4493,0.5243,0.0,0.3740
1,cohort_fe_plus_effort,2739,0.4960,0.0203,0.4561,0.5358,0.0,0.3814
2,mid2_controls_for_mid1,1346,0.1618,0.0240,0.1148,0.2088,0.0,0.5208


Coefficients are exam points per Parsons point; ×10 gives points per 10 Parsons points.

## 4. Quartile tables for the dashboard

In [7]:
show(stats.quartile_table)
quartiles = pd.concat([stats.quartile_table(features, c) for c in pipeline.HEADLINE_FEATURES if c in features], ignore_index=True)
quartiles.to_csv(T / "feature_quartile_scores.csv", index=False)
quartiles.loc[quartiles["feature"].eq(stats.PARSONS)].groupby("feature_quartile")[["mean_score_pct"]].mean().round(1)

`so_ell.stats.quartile_table`
```python
def quartile_table(df: pd.DataFrame, feature: str, target: str = "score_pct") -> pd.DataFrame:
    use = add_cohort(df)[["cohort", "semester_raw", "midterm", feature, target]].dropna().copy()
    use["feature_rank"] = use.groupby("cohort")[feature].rank(pct=True, method="average")
    use["feature_quartile"] = pd.cut(
        use["feature_rank"], bins=[0, 0.25, 0.5, 0.75, 1.0], labels=["Q1 bottom", "Q2", "Q3", "Q4 top"], include_lowest=True
    )
    summary = (
        use.groupby(["semester_raw", "midterm", "feature_quartile"], observed=False)
        .agg(n=(target, "size"), mean_score_pct=(target, "mean"))
        .reset_index()
    )
    summary["feature"] = feature
    return summary
```

,mean_score_pct
feature_quartile,
Q1 bottom,59.5
Q2,66.4
Q3,72.7
Q4 top,79.5


## 5. Change between midterms
Did students who changed their practice between midterm 1 and 2 change their score?

In [8]:
show(mid1_to_mid2_changes, pipeline.parsons_change_groups)
change = mid1_to_mid2_changes(features)
change_cols = [c for c in change.columns if c.endswith("__change")]
stats.compute_corr_table(change, "semester_raw", "score_change", change_cols).to_csv(T / "change_signals.csv", index=False)
groups = pipeline.parsons_change_groups(change)
groups.to_csv(T / "parsons_change_groups.csv", index=False)
groups.round(2)

`so_ell.features.mid1_to_mid2_changes`
```python
def mid1_to_mid2_changes(features: pd.DataFrame) -> pd.DataFrame:
    """One row per student who took both midterms, with mid1, mid2 and change columns."""
    numeric = [c for c in features.columns if "__" in c and pd.api.types.is_numeric_dtype(features[c])]
    base = features[["semester_raw", "anon_student_id", "midterm", "score_pct"] + numeric]
    wide = base.set_index(["semester_raw", "anon_student_id", "midterm"]).unstack("midterm")
    wide.columns = [f"{metric}__{midterm}" for metric, midterm in wide.columns]
    wide = wide.reset_index()
    wide = wide.loc[wide["score_pct__mid1"].notna() & wide["score_pct__mid2"].notna()].copy()
    wide["score_change"] = wide["score_pct__mid2"] - wide["score_pct__mid1"]
    changes = {
        f"{f}__change": wide[f"{f}__mid2"] - wide[f"{f}__mid1"]
        for f in numeric
        if f"{f}__mid1" in wide and f"{f}__mid2" in wide
    }
    return pd.concat([wide, pd.DataFrame(changes, index=wide.index)], axis=1)
```

`so_ell.pipeline.parsons_change_groups`
```python
def parsons_change_groups(change: pd.DataFrame, min_group: int = 10) -> pd.DataFrame:
    """Students split by Parsons first-try success before mid1 (above/below their
    semester's median) and by whether they did new Parsons problems before mid2.

    Groups smaller than ``min_group`` students are suppressed so no published
    row describes an individual.
    """
    use = change.copy()
    first = use["parsons__mean_first_score__mid1"]
    median = first.groupby(use["semester_raw"]).transform("median")
    use["baseline_group"] = "no_parsons_before_mid1"
    use.loc[first.notna() & first.lt(median), "baseline_group"] = "below_median_first_try_before_mid1"
    use.loc[first.notna() & first.ge(median), "baseline_group"] = "above_median_first_try_before_mid1"
    # Features are cumulative, so "change" is the practice done between the exams.
    use["usage_group"] = use["parsons__unique_problems__change"].fillna(0).gt(0).map(
        {True: "did_more_parsons_before_mid2", False: "no_new_parsons_before_mid2"}
    )
    out = (
        use.groupby(["baseline_group", "usage_group"], as_index=False)
        .agg(
            n_students=("anon_student_id", "count"),
            mean_mid1_pct=("score_pct__mid1", "mean"),
            mean_mid2_pct=("score_pct__mid2", "mean"),
            mean_score_change=("score_change", "mean"),
        )
    )
    return out.loc[out["n_students"] >= min_group].reset_index(drop=True)
```

,baseline_group,usage_group,n_students,mean_mid1_pct,mean_mid2_pct,mean_score_change
0,above_median_first_try_before_mid1,did_more_parsons_before_mid2,682,80.29,71.05,-9.24
1,below_median_first_try_before_mid1,did_more_parsons_before_mid2,662,65.22,59.22,-6.00


## 6. Sensitivity to the inferred answer keys
Re-score every exam at each key threshold (1.01 = drop inferred keys entirely) and refit.

In [9]:
show(pipeline.key_threshold_sensitivity)
sens = pipeline.key_threshold_sensitivity(features)
sens.to_csv(T / "key_threshold_sensitivity.csv", index=False)
sens.round(3)

`so_ell.pipeline.key_threshold_sensitivity`
```python
def key_threshold_sensitivity(features: pd.DataFrame) -> pd.DataFrame:
    """Does the headline survive a different answer-key threshold, or none at all?"""
    windows = _windows()
    raw = load_scoreable_events()
    rows = []
    for threshold in [0.55, 0.65, 0.75, 0.85, 1.01]:
        scores = score_exams(raw, windows, key_threshold=threshold)["student_scores"]
        swapped = features.drop(columns=["score_pct"]).merge(scores[WINDOW_KEYS + ["score_pct"]], on=WINDOW_KEYS)
        fe = stats.fit_parsons_models(swapped).iloc[0]
        rows.append(
            {
                "key_threshold": threshold,
                "label": "inferred keys dropped" if threshold > 1 else f"keys need {threshold:.0%} agreement",
                "n_student_exams": len(swapped),
                "parsons_quartile_gap": stats.within_group_quartile_gap(swapped, "cohort", stats.PARSONS, "score_pct"),
                "parsons_rank_corr": stats.within_group_rank_corr(swapped, "cohort", stats.PARSONS, "score_pct"),
                "fe_coef": fe["coef"],
                "fe_ci_low": fe["ci_low"],
                "fe_ci_high": fe["ci_high"],
            }
        )
        _log(f"analysis: key threshold {threshold} done")
    return pd.DataFrame(rows)
```

[21:32:13] analysis: key threshold 0.55 done


[21:34:11] analysis: key threshold 0.65 done


[21:36:09] analysis: key threshold 0.75 done


[21:38:07] analysis: key threshold 0.85 done


[21:40:05] analysis: key threshold 1.01 done


,key_threshold,label,n_student_exams,parsons_quartile_gap,parsons_rank_corr,fe_coef,fe_ci_low,fe_ci_high
0,0.55,keys need 55% agreement,2745,20.350,0.570,0.486,0.448,0.523
1,0.65,keys need 65% agreement,2745,20.390,0.570,0.487,0.449,0.524
2,0.75,keys need 75% agreement,2745,20.285,0.571,0.483,0.446,0.520
3,0.85,keys need 85% agreement,2745,20.308,0.572,0.484,0.447,0.521
4,1.01,inferred keys dropped,2745,20.503,0.571,0.489,0.452,0.527
